In [2]:
%run "////home/usriniva/Desktop/Phd/Epigenetics/Simulations/New_Hanna_eqns/Hanna_simu/simulation/simulation_prereqs/impport_packages.ipynb"    #import all necessary packages - numpy, pandas etc
%run "////home/usriniva/Desktop/Phd/Epigenetics/Simulations/New_Hanna_eqns/Hanna_simu/simulation/simulation_prereqs/simulation_class.ipynb"    #import all necessary packages - numpy, pandas etc
%run "////home/usriniva/Desktop/Phd/Epigenetics/Simulations/New_Hanna_eqns/Hanna_simu/simulation/simulation_prereqs/create_world.ipynb"

### Reformulation

- Define the **maternal deviation**:


##### $\delta_t$= Phenotype of the mother - Genetic contribution of mother
##### $\delta_t$ = $P_{t-1} - G_{t-1}$

so that the phenotype is the sum of the genetic component of the current generation and maternal deviation from its phenotype:

$$
P = G_t + \omega * \delta_t + \varepsilon_{d}
$$

- We also separate mutation-related noise $\varepsilon_{m}$ from parental phenotype inheritance noise $\varepsilon_{epi}$.  
- We define a evolvable 'weight' term $\omega$: determines how much importance to give each term of the equation
- We also add developmental noise: purely environmental noise in teh form of  $\varepsilon_{d}$


The full new formulation becomes:

$$
Z' = (G_t + \varepsilon_{m}) + (\omega + \varepsilon_{epi})*(\delta_t) +  \varepsilon_{d}
$$

---

In [3]:
def run_simulation(args):
    env_param_space, popsize, maxgen, dims, N, mutsize, mutrate, epsilon, nof_dimensions, only_genetics = args
    
    rho_m_alpha_beta = env_param_space[0:4]
    env_states = env_param_space[4:]

    meanw = np.zeros(maxgen)
    meanmemory_g = np.zeros(maxgen)
    meanmemory_p= np.zeros(maxgen)
    meanneutral_g = np.zeros(maxgen)
    meanneutral_p = np.zeros(maxgen)
    mean_gen = np.zeros(maxgen)
    stdmemory_p = np.zeros(maxgen)
    stdneutral_p = np.zeros(maxgen)
    
    for t in range(0, (maxgen-1)):
        #if t%10 == 0:
        #    print(t)
        for d in range(0, nof_dimensions):
            N[:,dims['dev'][d]] = N[:,dims['pheno'][d]] - env_states[t]
            
        dev_combined = np.sqrt(np.sum(N[:,dims['dev']]**2, axis =1))     
        W = np.exp((-dev_combined**2)/ (2))
        
        
        meanw[t] = np.mean(W)    
        
        meanmemory_g[t] = np.mean(N[:, dims['memory']])  # the actual memory
        meanmemory_p[t] = np.mean(1/(1 + np.exp(-N[:,dims['memory']]))) #with the logistc correction
        
        meanneutral_g[t] = np.mean(N[:,dims['neutral']])
        meanneutral_p[t] = np.mean(1/(1 + np.exp(-N[:,dims['neutral']])))
        mean_gen[t] = np.mean(N[:,dims['geno']])
        
        stdmemory_p[t] = np.std(1/(1 + np.exp(-N[:,dims['memory']])))
        stdneutral_p[t] = np.std(1/(1 + np.exp(-N[:,dims['neutral']])))
        
        #current gen offspring (the cube gets stored here per gen)
        
        offspring = np.zeros((popsize, dims['total_layers'])) # empty 3D matrix with dimensions of the cube
        
        #sample offspring for each scenario weighted by fitness
        
        
        parents_idx = np.random.choice(popsize, size=popsize, p = (W/np.sum(W)) )#pick #popsize sized random numbers
        offspring[:,:] = N[parents_idx, :] #similar to how it works in matlab
        
        mutate_memories = np.random.uniform(low=0, high=1, size= popsize) < mutrate[0]
        #mutate_neutral = np.random.uniform(low=0, high=1, size=popsize) < mutrate[0]
        mutate_geno = np.random.uniform(low=0, high=1, size=(popsize, nof_dimensions)) < mutrate[1]
        
        # adding a mutation of size mutsize[1], rate mutate_memories, to the epi-memory
        # the epignetic memory evolves; 
        
        offspring[:,dims['memory']] = (
                                offspring[:,dims['memory']] 
                                + mutate_memories * np.random.normal(0, mutsize[0], popsize)
        )
        
        # adding a mutation of size mutsize[1] (epi-mutation size), rate mutate_memories, to the neutral phenotype
        # the neutral trait recieves random mutation but is not implemented in the calculation for fitness; so is evolving neutrally
        offspring[:,dims['neutral']] = (
                                offspring[:,dims['neutral']]  
                                + mutate_memories * np.random.normal(0, mutsize[0], popsize)
        ) 
        
        # adding a mutation of size mutsize[2] (genetic mutation size), rate mutate_memories, to the genotype

        offspring[:,dims['geno']] = (
                            offspring[:,dims['geno']] 
                            + mutate_geno * np.random.normal(0, mutsize[1], size=(popsize, nof_dimensions) )
        )
        
        if only_genetics :
            if t % 10000 == 0:
                print(f"Gen {t}: Running ONLY genetics mode")
            
            for i in range(0, nof_dimensions):
                offspring[:, dims['pheno'][i]] = (
                offspring[:, dims['geno'][i]]
                + np.random.normal(0, epsilon, size=popsize) )
        else:
            if t % 10000 == 0:
                print(f"Gen {t}: Running EPIGENETIC mode")
            
            for i in range(0, nof_dimensions):
                    offspring[:, dims['pheno'][i]] = (
                        offspring[:, dims['geno'][i]] #genotype
                        + np.random.normal(0, epsilon, size=popsize) # adding random noise (developemental noise, nromally distributed with epsilon variance)
                        + ((1 / (1 + np.exp(-offspring[:, dims['memory']]))) #phenotype version of the memory (y axis) 
                        * (N[parents_idx, dims['pheno'][i]] - N[parents_idx, dims['geno'][i]]))# adding the deviation -> read as weight (phenotypic component * deviation)
                    )
        N = offspring
        
        
    return {                ## Make sure this is indented!!
        'maxgen_popsize': [maxgen,popsize],
        'rho_m_alpha_beta' :rho_m_alpha_beta,
        'meanw': meanw,
        'meanmemory_g': meanmemory_g,
        'meanmemory_p': meanmemory_p,
        'meanneutral_g': meanneutral_g,
        'meanneutral_p': meanneutral_p,
        'mean_geno': mean_gen,
        'std_memory_p': stdmemory_p,
        'std_neutral_p': stdneutral_p
        }
        

In [ ]:
rho_cases = 10
m_cases=  10
maxgen= 10000


full_env_values = create_world(rho_cases, m_cases, maxgen)   # create world (input: generations, no. rhos and no. ms) 
print(full_env_values.shape)

clean_data_env = full_env_values[~np.isnan(full_env_values).any(axis=1)]
clean_data_env

(100, 10004)


array([[-7.700e-01,  5.100e-01,  8.673e-01, ...,  0.000e+00,  1.000e+00,
         0.000e+00],
       [-5.500e-01,  5.100e-01,  7.595e-01, ...,  1.000e+00,  1.000e+00,
         1.000e+00],
       [-3.300e-01,  5.100e-01,  6.517e-01, ...,  1.000e+00,  0.000e+00,
         1.000e+00],
       ...,
       [ 5.500e-01,  9.900e-01,  4.500e-03, ...,  0.000e+00,  0.000e+00,
         0.000e+00],
       [ 7.700e-01,  9.900e-01,  2.300e-03, ...,  0.000e+00,  0.000e+00,
         0.000e+00],
       [ 9.900e-01,  9.900e-01,  1.000e-04, ...,  1.000e+00,  1.000e+00,
         1.000e+00]], shape=(68, 10004))

In [4]:

nof_scenarios = np.shape(clean_data_env)[0]
print('nof_scenarios:', nof_scenarios) 

rho_m_alpha_beta = clean_data_env[0:4]
env_states = clean_data_env[4:]


nof_scenarios: 68


In [5]:
popsize = 2500

# Mutation rates: [epi-memory mutation rate, geno mutation rate]
mutrate = [0.001, 0.001]

# Mutation sizes: [epi-memory mutation size, geno mutation size]
mutsize = [1, 1]

epsilon = 0.05 # phenotypic noise
                                 
initial_memory = 0

results_array = {}  ### DONT FORGET TO RUN BEFORE NEW PARAM CONDITIONS

mutrate[0]*mutsize[0]

0.001

 mutrate = [0.0001, 0.0001] seems to produce mutation sizes that are good enouhgh for for attaining high values and not fluctuating a lot because of too much highness

In [9]:
onlygen = False

# Number of dimensions (can be 1 or more)
nof_dimensions = 1

# get the cube dimensions
dims = get_cube_dims(nof_dimensions)

#create the cube
N = create_cube(popsize, dims=dims, nof_dimensions=nof_dimensions, epsilon=epsilon, initial_memory=initial_memory)

print(N.shape)


(2500, 5)


In [10]:

if __name__ == '__main__':
    
    # Create all combinations of sigma_mut and sigma_alpha
    param_grid = [
        (scenario, popsize, maxgen, dims, N, mutsize, mutrate, epsilon, nof_dimensions, onlygen) #only genetics can either be True or false
        for _, scenario in enumerate(clean_data_env) #where each row corresponds to one scenario
    ]

    num_cpus = 6
    
    with concurrent.futures.ProcessPoolExecutor(num_cpus) as executor:
        if onlygen:
            name = f'nof_d{nof_dimensions}_og' 
        else:
            name =f'nof_d{nof_dimensions}_epi'
        
        results_array = list(executor.map(run_simulation, param_grid))
        #results_array[name] = list(executor.map(run_simulation, param_grid))




Gen 0: Running EPIGENETIC modeGen 0: Running EPIGENETIC modeGen 0: Running EPIGENETIC modeGen 0: Running EPIGENETIC mode


Gen 0: Running EPIGENETIC modeGen 0: Running EPIGENETIC mode


Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: Running EPIGENETIC mode
Gen 0: R

In [11]:
folder_path='/home/usriniva/Desktop/Phd/Epigenetics/Simulations/New_Hanna_eqns/Hanna_simu/results/new_results/mutrate_0.001/'

# Choose start folder based on initial_memory
if initial_memory == -3:
    start = 'start-3'
elif initial_memory == 0:
    start = 'start0'
elif initial_memory == 3:
    start = 'start3'
else:
    raise ValueError(f"Unexpected initial_memory value: {initial_memory}")

# Build filename and full path
if onlygen:
    case = 'onlygen'
else:
    case = 'epi'

filename = f'gens{maxgen}_start{initial_memory}_mutrate{mutrate[0]}_dims{nof_dimensions}.pkl'


full_path = os.path.join(folder_path, start, case, filename)

# Save file
with open(full_path, 'wb') as f:
    pickle.dump(results_array, f)


THINK BEFORE RUNNING BELOW!!!!

##### open close pkl files

In [27]:
initial_memory  = 2
only_gen = False

In [14]:
param_grid = [
        (scenario, popsize, maxgen, dims, mutsize, mutrate, epsilon, nof_dimensions,initial_mem, only_gen) #only genetics can either be True or false
        for _, scenario in enumerate(clean_data_env) #where each row corresponds to one scenario
        for _, initial_mem in enumerate(initial_memory) #where each row corresponds to one initial mem
        
    ]



In [28]:
start = chr(initial_memory)

start

'\x02'